# Notebook 3: Stratified Sampling Strategy

## Purpose
This notebook designs and executes a stratified sampling strategy to select 7,000 comments from the raw corpus, ensuring:
1. Article diversity
2. Temporal coverage (2012-2016)
3. Comment type balance (top-level vs replies)
4. Engagement level diversity (TotalVotes)

## Overview
- **Input**: gnm_comments.csv (raw corpus)
- **Output**: sampled_7000_comments.csv with all metadata


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

base_path = Path('../archive/SOCC')
raw_comments_path = base_path / 'raw/gnm_comments.csv'
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)


## Section 1: Sampling Strategy Design

Stratified sampling across:
1. Article diversity
2. Temporal coverage (2012-2016)
3. Comment type (top-level vs replies)
4. Engagement levels (TotalVotes)

Target: 7,000 comments


In [2]:
TARGET_SAMPLES = 7000
DATE_START = 2012
DATE_END = 2016


## Section 2: Load and Prepare Raw Data

Since the raw comments file is large (>200MB), we'll process it in chunks and apply filters early to reduce memory usage.


In [3]:
chunk_iterator = pd.read_csv(raw_comments_path, chunksize=10000)
first_chunk = next(chunk_iterator)
print(f"Columns: {len(first_chunk.columns)}")


Columns: 28


In [4]:
filtered_chunks = []
chunk_iterator = pd.read_csv(raw_comments_path, chunksize=10000)

total_processed = 0
for i, chunk in enumerate(tqdm(chunk_iterator, desc="Processing chunks")):
    total_processed += len(chunk)
    
    required_cols = ['article_id', 'comment_text', 'timestamp', 'comment_id']
    chunk = chunk.dropna(subset=required_cols).copy()
    
    if len(chunk) == 0:
        continue
    
    chunk['datetime'] = pd.to_datetime(chunk['timestamp'], unit='ms', errors='coerce')
    chunk['year'] = chunk['datetime'].dt.year
    
    chunk = chunk[(chunk['year'] >= DATE_START) & (chunk['year'] <= DATE_END)].copy()
    
    if len(chunk) == 0:
        continue
    
    chunk['is_top_level'] = chunk['parentID'].isna() | (chunk['parentID'] == '')
    chunk['TotalVotes'] = pd.to_numeric(chunk['TotalVotes'], errors='coerce').fillna(0)
    
    chunk['engagement_bin'] = pd.cut(
        chunk['TotalVotes'],
        bins=[-1, 0, 5, 20, np.inf],
        labels=['low', 'med', 'high', 'very_high']
    )
    
    keep_cols = ['article_id', 'comment_counter', 'comment_id', 'comment_text', 
                 'comment_author', 'timestamp', 'datetime', 'year',
                 'parentID', 'is_top_level', 'TotalVotes', 'posVotes', 'negVotes', 'vote',
                 'engagement_bin', 'threadID', 'replies', 'descendantsCount', 'sender_isSelf']
    
    available_cols = [col for col in keep_cols if col in chunk.columns]
    chunk = chunk[available_cols].copy()
    
    filtered_chunks.append(chunk)
    
    if (i + 1) % 50 == 0:
        current_total = sum(len(c) for c in filtered_chunks)
        print(f"Processed {i+1} chunks, {current_total:,} valid comments...")

print(f"Processed {total_processed:,} total rows, {len(filtered_chunks)} chunks with valid data")


Processing chunks: 0it [00:00, ?it/s]

Processing chunks: 51it [00:05, 10.57it/s]

Processed 50 chunks, 500,000 valid comments...


Processing chunks: 67it [00:07,  9.51it/s]

Processed 663,173 total rows, 65 chunks with valid data


In [5]:
df_raw = pd.concat(filtered_chunks, ignore_index=True)
print(f"Combined dataframe: {len(df_raw):,} comments")
print(f"Columns: {len(df_raw.columns)}")


Combined dataframe: 643,642 comments
Columns: 19


In [6]:
print("Data distribution:")
print(f"Year: {df_raw['year'].value_counts().sort_index().to_dict()}")
print(f"Comment type: {df_raw['is_top_level'].value_counts().to_dict()}")
print(f"Engagement bins: {df_raw['engagement_bin'].value_counts().sort_index().to_dict()}")
print(f"Unique articles: {df_raw['article_id'].nunique():,}")


Data distribution:
Year: {2013: 142264, 2014: 147031, 2015: 183529, 2016: 170818}
Comment type: {False: 380491, True: 263151}
Engagement bins: {'low': 197044, 'med': 208605, 'high': 81811, 'very_high': 18161}
Unique articles: 7,655


## Section 3: Execute Stratified Sampling

We'll use a proportional stratified sampling approach to ensure balanced representation across all dimensions.


In [7]:
def stratified_sample(df, n_samples, stratify_cols, min_samples_per_stratum=1):
    grouped = df.groupby(stratify_cols, group_keys=False)
    group_sizes = grouped.size()
    total_size = len(df)
    
    samples_per_group = {}
    allocated = 0
    
    for name, size in group_sizes.items():
        proportional = int((size / total_size) * n_samples)
        target = max(proportional, min_samples_per_stratum)
        target = min(target, size)
        samples_per_group[name] = target
        allocated += target
    
    if allocated != n_samples:
        diff = n_samples - allocated
        sorted_groups = sorted(samples_per_group.items(), key=lambda x: x[1], reverse=True)
        
        for i, (name, current) in enumerate(sorted_groups):
            if diff == 0:
                break
            group_size = group_sizes[name]
            if diff > 0:
                add = min(diff, group_size - current)
                samples_per_group[name] += add
                diff -= add
            else:
                remove = min(-diff, current - min_samples_per_stratum)
                samples_per_group[name] -= remove
                diff += remove
    
    sampled_dfs = []
    for name, n_group_samples in samples_per_group.items():
        group_df = grouped.get_group(name)
        if len(group_df) > n_group_samples:
            sampled_group = group_df.sample(n=n_group_samples, random_state=42)
        else:
            sampled_group = group_df
        sampled_dfs.append(sampled_group)
    
    result = pd.concat(sampled_dfs, ignore_index=True)
    
    if len(result) > n_samples:
        result = result.sample(n=n_samples, random_state=42).reset_index(drop=True)
    
    return result


In [8]:
stratify_cols = ['year', 'is_top_level', 'engagement_bin']

df_sampled = stratified_sample(
    df_raw, 
    n_samples=TARGET_SAMPLES,
    stratify_cols=stratify_cols,
    min_samples_per_stratum=1
)

print(f"Sampled: {len(df_sampled):,} comments")
print(f"Unique articles: {df_sampled['article_id'].nunique():,}")

if len(df_sampled) > TARGET_SAMPLES:
    df_sampled = df_sampled.sample(n=TARGET_SAMPLES, random_state=42).reset_index(drop=True)
    print(f"Adjusted to {TARGET_SAMPLES:,} samples")


Sampled: 7,000 comments
Unique articles: 3,248


## Section 4: Validate Sample Quality

Check that the sampled data meets our stratification requirements.


In [9]:
print(f"Total samples: {len(df_sampled):,}")
print(f"Year distribution: {df_sampled['year'].value_counts().sort_index().to_dict()}")
print(f"Comment type: {df_sampled['is_top_level'].value_counts().to_dict()}")
print(f"Engagement bins: {df_sampled['engagement_bin'].value_counts().sort_index().to_dict()}")
print(f"Unique articles: {df_sampled['article_id'].nunique():,}")


Total samples: 7,000
Year distribution: {2013: 2936, 2014: 1207, 2015: 1492, 2016: 1365}
Comment type: {False: 4793, True: 2207}
Engagement bins: {'low': 3655, 'med': 2265, 'high': 886, 'very_high': 194}
Unique articles: 3,248


In [10]:
text_lengths = df_sampled['comment_text'].str.len()
print(f"Text quality: avg={text_lengths.mean():.1f} chars, median={text_lengths.median():.1f} chars")
empty_count = (df_sampled['comment_text'].isna() | (df_sampled['comment_text'].str.strip() == '')).sum()
if empty_count > 0:
    print(f"Warning: {empty_count} comments have empty text")


Text quality: avg=317.4 chars, median=204.0 chars


## Section 5: Export Sampled Dataset

Save the sampled comments to CSV for use in the next notebook.


In [11]:
sampled_output_path = output_dir / 'sampled_7000_comments.csv'
df_sampled.to_csv(sampled_output_path, index=False)
print(f"Saved: {sampled_output_path} ({len(df_sampled):,} samples, {len(df_sampled.columns)} columns)")


Saved: ..\data\processed\sampled_7000_comments.csv (7,000 samples, 19 columns)


## Summary

### Key Outputs

- `sampled_7000_comments.csv`: 7,000 comments selected using stratified sampling

### Next Steps

- Use this dataset for batch annotation with Perspective API
